# CANOPUS - Téléchargement et extraction des données riomètre

**Tutoriel :** Ce tutoriel explique comment extraire les données riomètres CANOPUS depuis le portail de données ouvertes.  
**Mission et instrument :** CANOPUS (Canadian Auroral Network for the OPEN Program Unified Study)  
**Objectif scientifique :** Mesurer le champ magnétique terrestre pour étudier les événements de météorologie spatiale tels que les tempêtes géomagnétiques et les sous-tempêtes.  
**Configuration requise :** Accès à Internet.
**Niveau du tutoriel :** Intermédiaire

Les données CARISMA sont disponibles au format CSV et en données brutes sur le portail de données ouvertes de l'ASC. Les données brutes se trouvent [ici](https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub/carisma/) et les fichiers CSV sont disponibles [ici](https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub/carisma_csv/). De plus, les données CARISMA sont également hébergées par l'Université de l'Alberta sur [carisma.ca](https://www.carisma.ca/).

***

# Partie 1 : Téléchargement des données CANOPUS

Les données CARISMA/CANOPUS sont hébergées sur le [CSA Open Data Portal](https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub). Les fichiers peuvent être téléchargés par scrapage de contenu.

## 1.1 Téléchargement programmatique (scraping HTTP/HTTPS)

### Structure du répertoire FTP 

```
/users/OpenData_DonneesOPuvertes/pub/
|
|-- carisma_csv/                                    <-- données magnétométriques CARISMA (csv)
|
|-- CANOPUS_CSV/                                    <-- données riomètre CANOPUS (csv)
|   +-- old_canopus_riometer_format/                <-- ancien format riomètre
|
|-- carisma/                                        <-- fichiers CARISMA BRUTS (.tar)
|
|-- CANOPUS/                                        <-- fichiers CANOPUS bruts (.tar.gz)
```
Stations magnétométriques : BACK, CONT, DAWS, ESKI, FCHU, FSIM, FSMI, GILL, GULL, ISLL, MCMU, MSTK, PINA, RABB, RANK, TALO




### Explorer le serveur

In [ ]:
# %pip install pandas matplotlib

In [ ]:
import os 
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
from io import StringIO

# Le serveur de l'ASC peut renvoyer un certificat auto-signé — désactiver la vérification SSL
requests.packages.urllib3.disable_warnings()

BASE_URL = "https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub/"
RIO_BASE = BASE_URL + "CANOPUS/CANOPUS_CSV/"

In [ ]:
def list_directory(url):
    resp = requests.get(url, verify=False)
    
    links = re.findall(r'href="([^"]+)"', resp.text)

    # Filtrer les liens vers les répertoires parents et les chaînes de requête
    names = []
    for link in links:
        name = link.strip('/').split('/')[-1]
        if name and name != ".." and "?" not in link and link != "../":
            # Ignorer les liens qui pointent vers le répertoire parent
            if not link.startswith("/"):
                names.append(name)
    return names    


# Parcourir l'archive des données riomètre
print("Années de données riomètre disponibles :")
years = list_directory(RIO_BASE)
print(years)

# Lister les stations pour 2007
print("\nStations disponibles en 2007 :")
stations = list_directory(RIO_BASE + "2007/")
print(stations)

# Lister les premiers fichiers pour la station GILL
print("\n5 premiers fichiers pour la station GILL en 2007 :")
files = list_directory(RIO_BASE + "2007/GILL/")
print(files[:5])


### Partie 1 : Télécharger les données Riomètre

En plus des données magnétométriques (voir le tutoriel CARISMA Magnetometer Data), CANOPUS a également collecté des données RIOMETER.

#### 1.1 Télécharger par programmation

In [ ]:
def download_rio_file(station, date_str, save_dir="data/rio"):
    year = date_str[:4]
    remote_dir = f"{RIO_BASE}{year}/{station}/"

    os.makedirs(save_dir, exist_ok=True)

    try:
        # Parcourir l'index du répertoire pour trouver les fichiers correspondants
        files = list_directory(remote_dir)
        matching_files = [f for f in files if date_str in f]

        if not matching_files:
            print(f"Aucun fichier riomètre trouvé pour {station} le {date_str}")
            return None
        
        filename = matching_files[0]
        file_url = remote_dir + filename
        local_path = os.path.join(save_dir, filename)

        resp = requests.get(file_url, verify=False)
        with open(local_path, 'wb') as f:
            f.write(resp.content)
        print(f"Téléchargé {filename} dans : {local_path}")
        return local_path
    
    except Exception as e:
        print(f"Erreur lors du téléchargement pour {station} le {date_str} : {e}")
        return None

# Exemple : Télécharger le fichier RIO pour la station GILL le 19990106
rio_file = download_rio_file("GILL", "19990106")

#### 1.2 Téléchargement manuel

1. Visitez d'abord le [Portail de données ouvertes de l'ASC - Jeu de données CARISMA (CSV)](https://donnees-data.asc-csa.gc.ca/users/OpenData_DonneesOuvertes/pub/carisma_csv/mag/daily/)
2. Parcourez ou recherchez les fichiers dont vous avez besoin
3. Téléchargez-les et enregistrez-les localement
4. Placez les fichiers dans le dossier /data/rio/ de ce tutoriel

### Partie 2 : Charger et explorer les données riomètre

Les fichiers riomètre ont une structure spécifique avec des lignes de commentaire, des métadonnées de station et les données elles-mêmes.

**Aperçu du format des fichiers :**
```
#CANOPUS Riometer Data ----  gil     19990106  lat:56.4 long:265.4
#Baseline Version 1  ---  Summary Data has not been checked for errors
"#contact:  Eric Donovan (edonovan@ucalgary.ca), Emma Spanswick (elspansw@ucalgary.ca)"
#----------------------------------------------
 
date(dd/mm/yy),time (UT),Absorption (dB),Raw Signal (Volts)
06/01/99,00:01:53,0.009,1.573
06/01/99,00:01:58,0.292,1.487
06/01/99,00:02:02,0.016,1.570
```

Les champs clés sont :
- **Absorption (dB)** : Absorption du bruit cosmique — des valeurs plus élevées indiquent une activité ionosphérique plus importante
- **Signal brut (Volts)** : La mesure de tension brute du riomètre


In [ ]:
def load_rio_data(file_path):
    data_lines = []
    header_found = False

    with open(file_path, 'r') as f:
        for line in f:
            stripped = line.strip().strip('"')

            # Ignorer les lignes de commentaire et les lignes vides
            if stripped.startswith("#") or stripped == "":
                continue
            
            # Détecter la ligne d'en-tête (date...), puis commencer à lire les données
            if not header_found and 'date' in stripped.lower():
                header_found = True
                continue

            # Ligne de données
            data_lines.append(stripped)

    # Convertir les lignes de données en DataFrame
    df = pd.read_csv(
        StringIO("\n".join(data_lines)),
        names=["date", "time", "absorption_dB", "raw_signal_V"],
        header=None
    )

    # Combiner la date et l'heure en une seule colonne datetime
    df['datetime'] = pd.to_datetime(
        df['date'] + ' ' + df['time'], 
        format='%d/%m/%y %H:%M:%S'
    )

    return df

In [ ]:
# Charger et explorer une date de données riomètre 
rio_df = load_rio_data(rio_file)

print(f"\nForme du dataframe riomètre : {rio_df.shape}")
print(f"Plage temporelle : {rio_df['datetime'].min()} à {rio_df['datetime'].max()}")
print(f"\nStatistiques d'absorption (dB) :")
print(rio_df[["absorption_dB", "raw_signal_V"]].describe().round(3))

rio_df.head()

### Partie 3 : Visualiser les données riomètre

Tracer les données riomètre : absorption et signal brut

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(rio_df['datetime'], rio_df['absorption_dB'], color='purple', linewidth=0.5)
axes[0].set_ylabel('Absorption (dB)')
axes[0].set_title(f'CANOPUS Riometer Data - GILL Station on {rio_df["datetime"].dt.date.iloc[0]}')
axes[0].grid(True, alpha=0.3)

axes[1].plot(rio_df['datetime'], rio_df['raw_signal_V'], color='teal', linewidth=0.5)
axes[1].set_ylabel('Raw Signal (V)')
axes[1].set_xlabel('Time (UTC)')
axes[1].grid(True, alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
axes[-1].xaxis.set_major_locator(mdates.HourLocator(interval=3))

plt.tight_layout()
plt.savefig("riometer_plot.png", dpi=150, bbox_inches='tight')
plt.show()

print("Le tracé riomètre a été enregistré sous 'riometer_plot.png'.")